# Exploration vs exploitation: how an agent decides what to try next

_Reinforcement Learning_

**Pull the arm that looks best now, or one you barely know? Greedy gets stuck; smart exploration shrinks over time and keeps regret small.**

The core tension. You face several actions ("arms"), each paying an unknown random reward. You
       only ever observe the reward of the arm you actually pull &mdash; never the others. So every pull
       does two jobs at once, and they fight:

       
         
- Exploit: pull the arm that looks best given what you know now, to earn reward this turn.
         
- Explore: pull a less-certain arm to learn whether it is secretly better, sacrificing
         this turn's expected reward for information that helps every future turn.
       

---

This notebook is a practice scaffold for the **Exploration vs exploitation: how an agent decides what to try next** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — numpy

In [ ]:
# Colab-ready: pure numpy, no gym/torch needed.
import numpy as np

# ============================================================
# 1. THE k-ARMED BANDIT ENVIRONMENT
#    Each arm a has a fixed true mean mu_a (unknown to the agent).
#    Pulling arm a returns a noisy reward ~ Normal(mu_a, 1).
# ============================================================
rng = np.random.default_rng(5)
k = 10
true_means = rng.normal(0.0, 1.0, k)   # the hidden values
opt = true_means.max()                 # mu* : the best arm's mean (oracle)
T = 2000                               # horizon (number of pulls)

def pull(a, r):                        # sample a reward from arm a
    return r.normal(true_means[a], 1.0)

# ============================================================
# 2. ONE GENERIC BANDIT LOOP, four selection rules
#    Q[a] = running average estimate of mu_a ; N[a] = pull count.
#    We update Q incrementally:  Q <- Q + (reward - Q) / N.
#    Regret at step t = mu* - mu_{a_t}  (expected reward lost).
# ============================================================
def run(strategy, eps=0.1, c=2.0, opt_init=0.0, seed=0):
    r = np.random.default_rng(seed)
    Q = np.full(k, opt_init, dtype=float)   # optimistic start if opt_init>0
    N = np.zeros(k)
    cum_regret = np.zeros(T)
    total = 0.0
    for t in range(1, T + 1):
        # ---- choose an arm a_t ----
        if strategy == "greedy":
            a = int(np.argmax(Q))                       # always exploit
        elif strategy == "egreedy":
            if r.random() < eps:
                a = int(r.integers(k))                  # explore: random arm
            else:
                a = int(np.argmax(Q))                   # exploit
        elif strategy == "optimistic":
            a = int(np.argmax(Q))                       # greedy, but Q starts high
        elif strategy == "ucb":
            bonus = c * np.sqrt(np.log(t) / (N + 1e-9)) # optimism under uncertainty
            a = int(np.argmax(Q + bonus))
        # ---- pull, observe reward, update estimate ----
        reward = pull(a, r)
        N[a] += 1
        Q[a] += (reward - Q[a]) / N[a]                  # incremental mean
        # ---- accumulate regret ----
        total += opt - true_means[a]
        cum_regret[t - 1] = total
    return cum_regret

g  = run("greedy",                          seed=6)
e  = run("egreedy",  eps=0.1,               seed=7)
ucb = run("ucb",     c=2.0,                 seed=8)
oi = run("optimistic", opt_init=5.0,        seed=9)   # start values WAY above any real reward

# ============================================================
# 3. REPORT FINAL CUMULATIVE REGRET (lower is better)
# ============================================================
print(f"true means (rounded): {np.round(true_means, 2)}")
print(f"best arm mean mu* = {opt:.3f}\n")
print(f"  greedy           final regret = {g[-1]:8.1f}   (gets stuck -> linear)")
print(f"  epsilon-greedy   final regret = {e[-1]:8.1f}   (sub-linear)")
print(f"  optimistic-init  final regret = {oi[-1]:8.1f}   (forces early exploration)")
print(f"  UCB              final regret = {ucb[-1]:8.1f}   (lowest -> logarithmic)")
# Typical output (seed 5):
#   greedy           ~ 2448.8   epsilon-greedy ~ 411.9
#   optimistic-init  ~ small     UCB           ~ 163.6

## Visualize the data & results

_On a real 10-armed bandit, how does cumulative regret grow for greedy vs ε-greedy vs UCB over 2000 pulls? (Lower is better.)_

In [ ]:
import numpy as np

# ---- a REAL 10-armed Gaussian bandit, simulated in numpy ----
rng = np.random.default_rng(5)
k, T = 10, 2000
true_means = rng.normal(0.0, 1.0, k)   # hidden arm values
opt = true_means.max()                 # mu* : best arm's mean

def run(strategy, eps=0.1, c=2.0, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros(k); N = np.zeros(k)
    cum = np.zeros(T); total = 0.0
    for t in range(1, T + 1):
        if strategy == "greedy":
            a = int(np.argmax(Q))
        elif strategy == "egreedy":
            a = int(r.integers(k)) if r.random() < eps else int(np.argmax(Q))
        elif strategy == "ucb":
            a = int(np.argmax(Q + c * np.sqrt(np.log(t) / (N + 1e-9))))
        reward = r.normal(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (reward - Q[a]) / N[a]          # incremental mean estimate
        total += opt - true_means[a]            # regret = mu* - mu_{a_t}
        cum[t - 1] = total
    return cum

g  = run("greedy",  seed=6)
e  = run("egreedy", eps=0.1, seed=7)
u  = run("ucb",     c=2.0,   seed=8)

# subsample to 40 points for plotting
idx = np.linspace(0, T - 1, 40).astype(int)
for name, arr in [("greedy", g), ("egreedy", e), ("ucb", u)]:
    print(name, [[int(i + 1), round(float(arr[i]), 1)] for i in idx])
print(f"FINAL  greedy={g[-1]:.1f}  egreedy={e[-1]:.1f}  ucb={u[-1]:.1f}")
# FINAL  greedy=2448.8  egreedy=411.9  ucb=163.6

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** An agent uses pure greedy on a 5-arm bandit. After 8 pulls it has tried arm 3 twice (both unlucky, low rewards) and arm 1 four times (decent), and now pulls ONLY arm 1 for the next 10,000 steps. Arm 3 is actually the best arm. What went wrong and what is the regret behaviour?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that greedy always pulls $\arg\max_a \hat Q(a)$ and never pulls anything else once an arm leads. — _With no exploration, the agent only updates the arm it already favours; the others' estimates are frozen at their early, noisy values._
- Observe that arm 3's two unlucky pulls left $\hat Q(3)$ artificially low, below $\hat Q(1)$. — _Greedy now believes arm 1 is best and will never pull arm 3 again, so it can never collect the evidence that would correct the estimate._
- Track the per-step regret: every step pays $\mu^* - \mu_1 > 0$ because arm 3 (the true best) is never chosen. — _A fixed positive gap added every step makes cumulative regret grow in a straight line (linear) forever._

**Answer:** Greedy got stuck on a sub-optimal arm. Two unlucky early pulls of the truly-best arm 3 dropped its estimate below arm 1's; with no exploration the agent never revisits arm 3, so the mistake is never corrected. It pays the fixed gap $\mu^* - \mu_1$ every step, so cumulative regret grows linearly &mdash; the classic too-little-exploration failure. The fix: inject exploration (decaying &epsilon;, optimistic initial values, or a UCB bonus) so under-tried arms keep getting sampled.

</details>

**Problem 2.** You run &epsilon;-greedy with a fixed $\varepsilon = 0.1$ and it works, but a teammate insists on decaying &epsilon; over time. On a stationary bandit, why is decaying better, and what is a simple schedule?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Split the cost of fixed $\varepsilon$ into early vs late. — _Early, estimates are noisy so exploring is valuable; late, estimates are accurate so exploring a known-bad arm is almost pure wasted reward._
- Quantify the late waste: with constant $\varepsilon=0.1$, even after finding the best arm you still pull a random arm 10% of the time, forever. — _That keeps adding a small fixed amount of regret per step, so regret never fully flattens — it stays linear with a small slope._
- Decay $\varepsilon$ toward 0, e.g. $\varepsilon_t = \min(1, c/t)$ or $1/t$. — _You explore hard when it pays (early) and almost never when it doesn't (late), letting cumulative regret flatten toward logarithmic._

**Answer:** On a stationary problem a constant $\varepsilon$ keeps wasting a fraction $\varepsilon$ of pulls on random arms forever &mdash; even after the best arm is obvious &mdash; so regret keeps climbing with slope roughly $\varepsilon \cdot (\text{avg gap})$. Decaying &epsilon; (e.g. $\varepsilon_t = 1/t$) explores a lot early, when estimates are noisy and learning is cheap, and almost not at all late, when estimates are sharp. That matches the principle "exploration must shrink over time" and pushes regret from linear toward logarithmic. (Caveat: if rewards are non-stationary, keep a floor of exploration instead of decaying to zero.)

</details>

**Problem 3.** Compare how &epsilon;-greedy and UCB decide to explore. UCB has a confidence bonus $c\sqrt{\ln t / N(a)}$. Explain why UCB usually achieves lower regret, and what the bonus does as $N(a)$ grows.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Describe &epsilon;-greedy's exploration: with probability $\varepsilon$ pick a UNIFORMLY random arm. — _Uniform means an arm already known to be terrible is just as likely to be explored as a promising under-tried one — exploration is spent blindly._
- Describe UCB's exploration: it pulls $\arg\max_a[\hat Q(a) + c\sqrt{\ln t / N(a)}]$. — _The bonus is large only for arms with small $N(a)$ (few pulls), so exploration is directed at arms that are uncertain AND could plausibly be best — not at arms already known to be bad._
- Track the bonus as $N(a)$ grows. — _$\sqrt{\ln t / N(a)}$ shrinks toward 0 as an arm is pulled more, so a well-understood arm stops being explored; the slow $\ln t$ growth keeps no arm abandoned forever._

**Answer:** &epsilon;-greedy explores uniformly and blindly &mdash; it wastes its exploration budget on arms it already knows are bad. UCB explores directionally: the bonus $c\sqrt{\ln t / N(a)}$ is large only for rarely-pulled, uncertain arms, so it probes exactly the arms that might be the best and stops probing clearly-bad ones. As $N(a)$ grows the bonus shrinks to ~0, so confident arms are exploited; the $\ln t$ term re-widens it slowly so nothing is forgotten. This "optimism under uncertainty" gives UCB provably logarithmic $O(\ln T)$ regret, beating &epsilon;-greedy in most stationary problems.

</details>